In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier,
                              VotingClassifier, StackingClassifier)
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Setup visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### TASK 1

In [ ]:
# 1. Load the data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine = pd.read_csv(url, sep=";")

# 2. Create binary target: 1 = good (quality >= 7), 0 = not good
wine["quality_label"] = (wine["quality"] >= 7).astype(int)

# 3. Data Exploration
print(f"Dataset Shape: {wine.shape}")
print("\nClass Distribution of quality_label:")
print(wine["quality_label"].value_counts(normalize=True))

# 4. Prepare Features and Target
X = wine.drop(['quality', 'quality_label'], axis=1)
y = wine['quality_label']

# 5. Split Data (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 6. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Fit Baseline Models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

baseline_results = []

for name, model in models.items():
    # Use scaled data for LR and KNN, but DT doesn't strictly need it
    X_train_fit = X_train_scaled if name != "Decision Tree" else X_train
    X_test_fit = X_test_scaled if name != "Decision Tree" else X_test
    
    model.fit(X_train_fit, y_train)
    y_pred = model.predict(X_test_fit)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="binary")
    
    baseline_results.append({"Model": name, "Accuracy": acc, "F1 Score": f1})

# Display Results
baseline_df = pd.DataFrame(baseline_results)
print("\nBaseline Model Results:")
print(baseline_df)

### TASK 2

In [ ]:
# 1. Bagging Classifier
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100, 
    oob_score=True, 
    random_state=42
)
bagging.fit(X_train, y_train)

# 2. Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=100, 
    oob_score=True, 
    random_state=42
)
rf.fit(X_train, y_train)

# Evaluation function
def evaluate_ensemble(model, name, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    oob = model.oob_score_
    return {"Model": name, "Accuracy": acc, "F1 Score": f1, "OOB Score": oob}

bagging_res = evaluate_ensemble(bagging, "Bagging (DT)", X_test, y_test)
rf_res = evaluate_ensemble(rf, "Random Forest", X_test, y_test)

print(pd.DataFrame([bagging_res, rf_res]))

# Feature Importance Plot
importances = rf.feature_importances_
indices = np.argsort(importances)[-10:]

plt.figure(figsize=(10, 6))
plt.title('Top 10 Feature Importances (Random Forest)')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

# Why Random Forest is better than a single Decision Tree

A single Decision Tree is highly prone to **overfitting**, meaning it learns the training data too well but performs poorly on unseen data.

**Random Forest** improves this by combining multiple decision trees and introducing two types of randomness:

---

## 1. Bootstrap Aggregating (Bagging)
- Each tree is trained on a **random subset of the original dataset**.
- Sampling is done **with replacement** (some samples may appear multiple times, others not at all).
- As a result, each tree sees a different version of the data.

---

## 2. Feature Randomness
- At each split in a tree, only a **random subset of features** is considered instead of all features.
- This prevents trees from always choosing the same dominant features.

---

## Result
Because of these two sources of randomness:

- Correlation between trees is reduced  
- The model becomes more stable  
- Overfitting is reduced  
- Generalization performance improves  

---

## Key Idea
> Random Forest = an ensemble of many diverse decision trees whose combined prediction is more accurate and robust than any single tree.

### TASK 3

In [ ]:
# 1. Train Boosting Models
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

ada.fit(X_train_scaled, y_train)
gb.fit(X_train_scaled, y_train)

boosting_results = []
for name, model in [("AdaBoost", ada), ("Gradient Boosting", gb)]:
    y_pred = model.predict(X_test_scaled)
    boosting_results.append({
        "Model": name, 
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

print(pd.DataFrame(boosting_results))

# 2. Learning Curves for Gradient Boosting
test_score = np.zeros((100,), dtype=np.float64)
train_score = np.zeros((100,), dtype=np.float64)

for i, y_pred in enumerate(gb.staged_predict(X_test_scaled)):
    test_score[i] = accuracy_score(y_test, y_pred)

for i, y_pred in enumerate(gb.staged_predict(X_train_scaled)):
    train_score[i] = accuracy_score(y_train, y_pred)

plt.figure(figsize=(10, 5))
plt.plot(np.arange(100) + 1, train_score, label='Train Accuracy')
plt.plot(np.arange(100) + 1, test_score, label='Test Accuracy')
plt.title('Gradient Boosting: Learning Curve')
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# AdaBoost vs Gradient Boosting

Both **AdaBoost** and **Gradient Boosting** are ensemble learning methods that build strong models by combining multiple weak learners (usually decision trees). However, they differ in how they improve mistakes.

---

## AdaBoost (Adaptive Boosting)
- Focuses on **misclassified samples**
- Increases the **weights** of incorrectly predicted data points
- Forces the next model to pay more attention to difficult samples
- Each new model tries to correct the mistakes of the previous ones by emphasizing hard examples

---

## Gradient Boosting
- Focuses on **prediction errors (residuals)**
- Each new model is trained to predict the **residual errors** of the previous model
- Uses the idea of **gradient descent** to minimize the loss function step by step
- Instead of changing sample weights, it directly improves error correction

---

## Key Difference
- **AdaBoost:** Adjusts sample weights to focus on hard-to-classify examples  
- **Gradient Boosting:** Fits models to the residual errors of previous models  

---

## Model Choice
- **Gradient Boosting** is generally more powerful and flexible  
  - Works well with complex datasets  
  - Supports many hyperparameter tuning options  
- **AdaBoost** is simpler and can be faster  
  - Performs well on smaller or less complex problems  

---

## Summary
> AdaBoost focuses on *hard samples*, while Gradient Boosting focuses on *reducing errors directly using optimization (gradient descent)*.

### TASK 4

In [ ]:
# 1. Best 3 models from previous tasks
best_rf = RandomForestClassifier(n_estimators=100, random_state=42)
best_gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
best_lr = LogisticRegression(max_iter=1000, random_state=42)

estimators = [
    ('rf', best_rf),
    ('gb', best_gb),
    ('lr', best_lr)
]

# 2. Voting Classifier (Soft)
voting = VotingClassifier(estimators=estimators, voting='soft')
voting.fit(X_train_scaled, y_train)

# 3. Stacking Classifier
stacking = StackingClassifier(
    estimators=estimators, 
    final_estimator=LogisticRegression(),
    cv=5
)
stacking.fit(X_train_scaled, y_train)

# Collect all final results
final_results = []
all_models = {
    "Logistic Regression": best_lr,
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": best_rf,
    "AdaBoost": ada,
    "Gradient Boosting": best_gb,
    "Voting Classifier": voting,
    "Stacking Classifier": stacking
}

for name, model in all_models.items():
    # Note: Stacking/Voting/LR/Ada/GB fit on scaled data for consistency
    if name == "Decision Tree" or name == "Random Forest":
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    
    final_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

comparison_df = pd.DataFrame(final_results).sort_values(by="F1 Score", ascending=False)
print("\nFinal Comparison Table:")
print(comparison_df)

# Conclusion

## Best Strategy
For most **tabular datasets**, ensemble methods such as **Random Forest** or **Stacking Classifier** usually achieve the highest **F1 score** compared to a single model.

---

## Performance Improvement
Compared to a single Decision Tree:
- Random Forest and Boosting methods significantly improve both **Accuracy** and **F1 score**
- They reduce overfitting and provide more stable predictions

---

## Trade-offs

### 1. Complexity
- Ensemble models are more complex to build and tune
- They require careful hyperparameter optimization

### 2. Training Time
- Multiple trees are trained, so they require **more computational power and time**

### 3. Interpretability
- A single Decision Tree is easy to interpret and visualize
- However, a Random Forest (many trees combined) becomes a **black-box model**, making interpretation harder

---

## Recommendation
For real-world systems, **Random Forest** is often a strong default choice because:

- It provides high accuracy
- It is robust against overfitting
- It offers **Feature Importance**, which helps explain results  
  (e.g., identifying that features like **Alcohol level** strongly influence wine quality)

---

## Final Insight
> Ensemble methods trade interpretability and simplicity for significantly better performance and robustness.